# ============================================================
# PharmaLens AI
# 05_GTM_Intelligence.ipynb
# ============================================================
# GO-TO-MARKET INTELLIGENCE
#
# Objective:
# Transform pharmaceutical sales data into actionable
# Go-To-Market (GTM) intelligence.
#
# Main analytical areas:
# 1. Distribution Channel Intelligence
# 2. Channel Market Share
# 3. Channel Growth
# 4. Brand x Channel Performance
# 5. Manufacturer x Channel Performance
# 6. Therapeutic Class x Channel Analysis
# 7. Launch Channel Readiness
# 8. Price & Channel Strategy
# 9. GTM Opportunity Scoring
# 10. Strategic GTM Recommendations
# 11. Executive GTM Dashboard
# ============================================================

In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
df=  pd.read_excel ("IMS (2021-2025).xlsx")

MemoryError: Unable to allocate 6.94 MiB for an array with shape (909922,) and data type int64

In [ ]:
df = df.rename(columns={
    "Sector": "Distribution Channel",
    "ATC4": "Therapeutic Class",
    "Corporation": "Manufacturer",
    "Product": "Brand Name",
    "Pack": "Pack Size",
    "Launch Date": "Product Launch",
    "Strength": "Drug Strength",
    "Retail Price": "Selling Price",
    "NFC3": "Market Category",
    "Period": "Month",
    "Calendar Year": "Year",
    "Units": "Sales Units",
    "LC Value": "Sales Value"
})

In [ ]:
# ============================================================
# 3. VALIDATE DATA DICTIONARY
# ============================================================

EXPECTED_COLUMNS = [
    "Distribution Channel",
    "Therapeutic Class",
    "Manufacturer",
    "Brand Name",
    "Pack Size",
    "Product Launch",
    "Drug Strength",
    "Selling Price",
    "Market Category",
    "Month",
    "Year",
    "Sales Units",
    "Sales Value"
]

print("Dataset columns:")
print(list(df.columns))

missing_columns = [
    col for col in EXPECTED_COLUMNS
    if col not in df.columns
]

if missing_columns:
    print("\n❌ Missing columns:")
    for col in missing_columns:
        print("-", col)

    raise ValueError(
        "Column validation failed. "
        "Please check the renamed dataset columns."
    )

print("\n✅ All expected GTM columns are available.")

In [ ]:
# ============================================================
# 4. DATA PREPARATION
# ============================================================

gtm_df = df.copy()

# ------------------------------------------------------------
# Clean text columns
# ------------------------------------------------------------

TEXT_COLUMNS = [
    "Distribution Channel",
    "Therapeutic Class",
    "Manufacturer",
    "Brand Name",
    "Pack Size",
    "Drug Strength",
    "Market Category"
]

for col in TEXT_COLUMNS:

    gtm_df[col] = (
        gtm_df[col]
        .astype(str)
        .str.strip()
        .replace({
            "nan": np.nan,
            "None": np.nan,
            "": np.nan
        })
    )

# ------------------------------------------------------------
# Numeric columns
# ------------------------------------------------------------

NUMERIC_COLUMNS = [
    "Selling Price",
    "Year",
    "Sales Units",
    "Sales Value"
]

for col in NUMERIC_COLUMNS:
    gtm_df[col] = pd.to_numeric(
        gtm_df[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# Product launch date
# ------------------------------------------------------------

gtm_df["Product Launch"] = pd.to_datetime(
    gtm_df["Product Launch"],
    errors="coerce"
)

# ------------------------------------------------------------
# Month
# ------------------------------------------------------------

gtm_df["Month"] = pd.to_datetime(
    gtm_df["Month"],
    errors="coerce"
)

# ------------------------------------------------------------
# Remove impossible sales values
# ------------------------------------------------------------

gtm_df = gtm_df[
    (gtm_df["Sales Value"].fillna(0) >= 0) &
    (gtm_df["Sales Units"].fillna(0) >= 0)
].copy()

print("Prepared dataset shape:", gtm_df.shape)

display(gtm_df.head())

In [ ]:
# ============================================================
# 5. DATA QUALITY CHECK
# ============================================================

quality = pd.DataFrame({
    "Column": gtm_df.columns,
    "Missing_Count": gtm_df.isna().sum().values,
    "Missing_%": (
        gtm_df.isna().mean().values * 100
    ).round(2),
    "Unique_Values": [
        gtm_df[col].nunique(dropna=True)
        for col in gtm_df.columns
    ]
})

display(
    quality.sort_values(
        "Missing_%",
        ascending=False
    )
)

print("\nDuplicate rows:", gtm_df.duplicated().sum())
print("Rows:", len(gtm_df))

In [ ]:
# ============================================================
# 6. CREATE GTM TIME FEATURES
# ============================================================

# If Month is available, use it as the primary time dimension
if gtm_df["Month"].notna().sum() > 0:

    gtm_df["GTM_Month"] = gtm_df["Month"].dt.to_period("M")

    gtm_df["GTM_Year"] = (
        gtm_df["Month"].dt.year
    )

    gtm_df["GTM_Month_Number"] = (
        gtm_df["Month"].dt.month
    )

else:

    gtm_df["GTM_Year"] = gtm_df["Year"]

    gtm_df["GTM_Month_Number"] = np.nan

# ------------------------------------------------------------
# Revenue per unit
# ------------------------------------------------------------

gtm_df["Revenue_Per_Unit"] = np.where(
    gtm_df["Sales Units"] > 0,
    gtm_df["Sales Value"] / gtm_df["Sales Units"],
    np.nan
)

# ------------------------------------------------------------
# Launch age
# ------------------------------------------------------------

analysis_year = int(
    gtm_df["GTM_Year"].dropna().max()
)

gtm_df["Launch_Year"] = (
    gtm_df["Product Launch"].dt.year
)

gtm_df["Launch_Age"] = (
    analysis_year - gtm_df["Launch_Year"]
)

gtm_df["Launch_Age"] = (
    gtm_df["Launch_Age"].clip(lower=0)
)

print("Analysis year:", analysis_year)

In [ ]:
# ============================================================
# 7. GTM MARKET OVERVIEW
# ============================================================

total_sales = gtm_df["Sales Value"].sum()
total_units = gtm_df["Sales Units"].sum()

total_brands = gtm_df["Brand Name"].nunique()
total_manufacturers = gtm_df["Manufacturer"].nunique()
total_channels = gtm_df["Distribution Channel"].nunique()
total_therapeutic_classes = gtm_df["Therapeutic Class"].nunique()

gtm_overview = pd.DataFrame({
    "KPI": [
        "Total Sales Value",
        "Total Sales Units",
        "Number of Brands",
        "Number of Manufacturers",
        "Number of Distribution Channels",
        "Number of Therapeutic Classes"
    ],
    "Value": [
        total_sales,
        total_units,
        total_brands,
        total_manufacturers,
        total_channels,
        total_therapeutic_classes
    ]
})

display(gtm_overview)

print("\nGTM Market Overview")
print("=" * 50)
print(f"Sales Value:        {total_sales:,.0f}")
print(f"Sales Units:        {total_units:,.0f}")
print(f"Brands:             {total_brands:,}")
print(f"Manufacturers:      {total_manufacturers:,}")
print(f"Channels:           {total_channels:,}")
print(f"Therapeutic Class:  {total_therapeutic_classes:,}")

In [ ]:
# ============================================================
# 8. DISTRIBUTION CHANNEL INTELLIGENCE
# ============================================================

channel_intelligence = (
    gtm_df
    .groupby("Distribution Channel")
    .agg(
        Sales_Value=("Sales Value", "sum"),
        Sales_Units=("Sales Units", "sum"),
        Brands=("Brand Name", "nunique"),
        Manufacturers=("Manufacturer", "nunique"),
        Therapeutic_Classes=("Therapeutic Class", "nunique"),
        Avg_Selling_Price=("Selling Price", "mean")
    )
    .reset_index()
)

channel_intelligence["Sales_Share_%"] = (
    channel_intelligence["Sales_Value"]
    / channel_intelligence["Sales_Value"].sum()
    * 100
)

channel_intelligence["Unit_Share_%"] = (
    channel_intelligence["Sales_Units"]
    / channel_intelligence["Sales_Units"].sum()
    * 100
)

channel_intelligence["Revenue_Per_Unit"] = np.where(
    channel_intelligence["Sales_Units"] > 0,
    channel_intelligence["Sales_Value"]
    / channel_intelligence["Sales_Units"],
    np.nan
)

channel_intelligence = (
    channel_intelligence
    .sort_values(
        "Sales_Value",
        ascending=False
    )
    .reset_index(drop=True)
)

display(channel_intelligence)

In [ ]:
# ============================================================
# 9. CHANNEL MARKET SHARE
# ============================================================

plt.figure(figsize=(10, 6))

plt.bar(
    channel_intelligence["Distribution Channel"],
    channel_intelligence["Sales_Share_%"]
)

plt.title(
    "Pharmaceutical Market Share by Distribution Channel"
)

plt.xlabel("Distribution Channel")
plt.ylabel("Sales Share (%)")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 10. CHANNEL SALES TREND
# ============================================================

if gtm_df["Month"].notna().sum() > 0:

    channel_monthly = (
        gtm_df
        .groupby([
            "Month",
            "Distribution Channel"
        ])["Sales Value"]
        .sum()
        .reset_index()
    )

    plt.figure(figsize=(14, 7))

    for channel in channel_monthly[
        "Distribution Channel"
    ].dropna().unique():

        temp = channel_monthly[
            channel_monthly[
                "Distribution Channel"
            ] == channel
        ]

        plt.plot(
            temp["Month"],
            temp["Sales Value"],
            label=channel
        )

    plt.title("Monthly Sales Trend by Distribution Channel")
    plt.xlabel("Month")
    plt.ylabel("Sales Value")
    plt.legend()
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

else:

    print("Monthly date information is not available.")

In [ ]:
# ============================================================
# 11. CHANNEL GROWTH ANALYSIS
# ============================================================

if gtm_df["GTM_Year"].notna().sum() > 0:

    channel_yearly = (
        gtm_df
        .groupby([
            "GTM_Year",
            "Distribution Channel"
        ])["Sales Value"]
        .sum()
        .reset_index()
    )

    channel_growth = (
        channel_yearly
        .sort_values([
            "Distribution Channel",
            "GTM_Year"
        ])
        .copy()
    )

    channel_growth["YoY_Growth_%"] = (
        channel_growth
        .groupby("Distribution Channel")["Sales Value"]
        .pct_change()
        * 100
    )

    display(channel_growth)

else:

    channel_growth = pd.DataFrame()
    print("Year information unavailable.")

In [ ]:
# ============================================================
# 12. BRAND x CHANNEL INTELLIGENCE
# ============================================================

brand_channel = (
    gtm_df
    .groupby([
        "Brand Name",
        "Distribution Channel"
    ])
    .agg(
        Sales_Value=("Sales Value", "sum"),
        Sales_Units=("Sales Units", "sum"),
        Avg_Price=("Selling Price", "mean"),
        Manufacturers=("Manufacturer", "nunique"),
        Therapeutic_Classes=("Therapeutic Class", "nunique")
    )
    .reset_index()
)

brand_channel["Channel_Share_%"] = (
    brand_channel
    .groupby("Brand Name")["Sales_Value"]
    .transform(
        lambda x: x / x.sum() * 100
    )
)

brand_channel = (
    brand_channel
    .sort_values(
        "Sales_Value",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Top Brand x Channel combinations")

display(
    brand_channel.head(30)
)

In [ ]:
# ============================================================
# 12. BRAND x CHANNEL INTELLIGENCE
# ============================================================

brand_channel = (
    gtm_df
    .groupby([
        "Brand Name",
        "Distribution Channel"
    ])
    .agg(
        Sales_Value=("Sales Value", "sum"),
        Sales_Units=("Sales Units", "sum"),
        Avg_Price=("Selling Price", "mean"),
        Manufacturers=("Manufacturer", "nunique"),
        Therapeutic_Classes=("Therapeutic Class", "nunique")
    )
    .reset_index()
)

brand_channel["Channel_Share_%"] = (
    brand_channel
    .groupby("Brand Name")["Sales_Value"]
    .transform(
        lambda x: x / x.sum() * 100
    )
)

brand_channel = (
    brand_channel
    .sort_values(
        "Sales_Value",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Top Brand x Channel combinations")

display(
    brand_channel.head(30)
)

In [ ]:
# ============================================================
# 13. TOP BRANDS WITHIN EACH CHANNEL
# ============================================================

top_brands_by_channel = (
    brand_channel
    .sort_values(
        [
            "Distribution Channel",
            "Sales_Value"
        ],
        ascending=[True, False]
    )
    .groupby("Distribution Channel")
    .head(10)
    .reset_index(drop=True)
)

display(top_brands_by_channel)

In [ ]:
# ============================================================
# 14. MANUFACTURER x CHANNEL INTELLIGENCE
# ============================================================

manufacturer_channel = (
    gtm_df
    .groupby([
        "Manufacturer",
        "Distribution Channel"
    ])
    .agg(
        Sales_Value=("Sales Value", "sum"),
        Sales_Units=("Sales Units", "sum"),
        Brands=("Brand Name", "nunique"),
        Avg_Price=("Selling Price", "mean")
    )
    .reset_index()
)

manufacturer_channel["Channel_Share_%"] = (
    manufacturer_channel
    .groupby("Manufacturer")["Sales_Value"]
    .transform(
        lambda x: x / x.sum() * 100
    )
)

manufacturer_channel = (
    manufacturer_channel
    .sort_values(
        "Sales_Value",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    manufacturer_channel.head(30)
)

In [ ]:
# ============================================================
# 15. THERAPEUTIC CLASS x CHANNEL INTELLIGENCE
# ============================================================

therapeutic_channel = (
    gtm_df
    .groupby([
        "Therapeutic Class",
        "Distribution Channel"
    ])
    .agg(
        Sales_Value=("Sales Value", "sum"),
        Sales_Units=("Sales Units", "sum"),
        Brands=("Brand Name", "nunique")
    )
    .reset_index()
)

therapeutic_channel["Channel_Share_%"] = (
    therapeutic_channel
    .groupby("Therapeutic Class")["Sales_Value"]
    .transform(
        lambda x: x / x.sum() * 100
    )
)

display(
    therapeutic_channel
    .sort_values(
        "Sales_Value",
        ascending=False
    )
    .head(50)
)

In [ ]:
# ============================================================
# 16. CHANNEL CONCENTRATION
# ============================================================

channel_concentration = (
    gtm_df
    .groupby("Distribution Channel")["Sales Value"]
    .sum()
    .sort_values(ascending=False)
)

channel_concentration_share = (
    channel_concentration
    / channel_concentration.sum()
)

top_channel_share = (
    channel_concentration_share
    .head(1)
    .sum()
    * 100
)

top_3_channel_share = (
    channel_concentration_share
    .head(3)
    .sum()
    * 100
)

print(
    f"Top channel sales concentration: "
    f"{top_channel_share:.2f}%"
)

print(
    f"Top 3 channel sales concentration: "
    f"{top_3_channel_share:.2f}%"
)

In [ ]:
# ============================================================
# 17. PRODUCT LAUNCH CHANNEL READINESS
# ============================================================

launch_channel = gtm_df[
    gtm_df["Product Launch"].notna()
].copy()

if len(launch_channel) > 0:

    launch_channel["Launch_Year"] = (
        launch_channel["Product Launch"].dt.year
    )

    launch_channel_summary = (
        launch_channel
        .groupby("Distribution Channel")
        .agg(
            Launches=("Brand Name", "nunique"),
            Sales_Value=("Sales Value", "sum"),
            Sales_Units=("Sales Units", "sum"),
            Brands=("Brand Name", "nunique")
        )
        .reset_index()
    )

    launch_channel_summary["Sales_per_Launch"] = np.where(
        launch_channel_summary["Launches"] > 0,
        launch_channel_summary["Sales_Value"]
        / launch_channel_summary["Launches"],
        np.nan
    )

    launch_channel_summary = (
        launch_channel_summary
        .sort_values(
            "Sales_per_Launch",
            ascending=False
        )
    )

    display(launch_channel_summary)

else:

    launch_channel_summary = pd.DataFrame()

    print(
        "Product Launch dates are not sufficiently available "
        "for launch-channel analysis."
    )

In [ ]:
# ============================================================
# 18. NEW PRODUCT LAUNCH PERFORMANCE
# ============================================================

if len(launch_channel) > 0:

    launch_channel["Years_Since_Launch"] = (
        analysis_year
        - launch_channel["Launch_Year"]
    )

    # New products = launched during the last 2 years
    new_product_data = launch_channel[
        launch_channel["Years_Since_Launch"].between(
            0, 2
        )
    ].copy()

    new_product_channel = (
        new_product_data
        .groupby("Distribution Channel")
        .agg(
            New_Product_Sales=(
                "Sales Value",
                "sum"
            ),
            New_Product_Units=(
                "Sales Units",
                "sum"
            ),
            New_Product_Brands=(
                "Brand Name",
                "nunique"
            )
        )
        .reset_index()
    )

    new_product_channel["Sales_per_New_Brand"] = np.where(
        new_product_channel["New_Product_Brands"] > 0,
        new_product_channel["New_Product_Sales"]
        / new_product_channel["New_Product_Brands"],
        np.nan
    )

    display(
        new_product_channel.sort_values(
            "New_Product_Sales",
            ascending=False
        )
    )

else:

    new_product_channel = pd.DataFrame()

In [ ]:
# ============================================================
# 19. PRICE & CHANNEL STRATEGY
# ============================================================

price_channel = (
    gtm_df
    .groupby("Distribution Channel")
    .agg(
        Avg_Selling_Price=("Selling Price", "mean"),
        Median_Selling_Price=("Selling Price", "median"),
        Sales_Value=("Sales Value", "sum"),
        Sales_Units=("Sales Units", "sum")
    )
    .reset_index()
)

price_channel["Revenue_Per_Unit"] = np.where(
    price_channel["Sales_Units"] > 0,
    price_channel["Sales_Value"]
    / price_channel["Sales_Units"],
    np.nan
)

# Relative price index
market_avg_price = (
    gtm_df["Selling Price"].mean()
)

price_channel["Price_Index"] = (
    price_channel["Avg_Selling_Price"]
    / market_avg_price
    * 100
)

display(
    price_channel.sort_values(
        "Sales_Value",
        ascending=False
    )
)

In [ ]:
# ============================================================
# 20. CHANNEL OPPORTUNITY MATRIX
# ============================================================

channel_strategy = channel_intelligence.copy()

# ------------------------------------------------------------
# Normalize performance metrics
# ------------------------------------------------------------

def minmax(series):

    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            50,
            index=series.index
        )

    return (
        (series - min_value)
        / (max_value - min_value)
        * 100
    )

channel_strategy["Scale_Score"] = minmax(
    channel_strategy["Sales_Value"]
)

channel_strategy["Brand_Coverage_Score"] = minmax(
    channel_strategy["Brands"]
)

channel_strategy["Manufacturer_Coverage_Score"] = minmax(
    channel_strategy["Manufacturers"]
)

channel_strategy["Unit_Score"] = minmax(
    channel_strategy["Sales_Units"]
)

# ------------------------------------------------------------
# GTM channel opportunity score
# ------------------------------------------------------------

channel_strategy["GTM_Opportunity_Score"] = (
    channel_strategy["Scale_Score"] * 0.40
    + channel_strategy["Brand_Coverage_Score"] * 0.20
    + channel_strategy["Manufacturer_Coverage_Score"] * 0.15
    + channel_strategy["Unit_Score"] * 0.25
)

# ------------------------------------------------------------
# Strategic classification
# ------------------------------------------------------------

channel_strategy["GTM_Priority"] = pd.cut(
    channel_strategy["GTM_Opportunity_Score"],
    bins=[-np.inf, 25, 50, 75, np.inf],
    labels=[
        "Low Priority",
        "Monitor",
        "Strategic Opportunity",
        "High Priority"
    ]
)

display(
    channel_strategy.sort_values(
        "GTM_Opportunity_Score",
        ascending=False
    )
)

In [ ]:
# ============================================================
# 21. GTM STRATEGIC ACTIONS
# ============================================================

def assign_gtm_action(priority):

    if priority == "High Priority":
        return "Scale & Invest"

    elif priority == "Strategic Opportunity":
        return "Expand & Validate"

    elif priority == "Monitor":
        return "Optimize & Monitor"

    else:
        return "Selective Coverage"


channel_strategy["GTM_Strategic_Action"] = (
    channel_strategy["GTM_Priority"]
    .astype(str)
    .apply(assign_gtm_action)
)

display(
    channel_strategy[
        [
            "Distribution Channel",
            "Sales_Value",
            "Sales_Share_%",
            "Brands",
            "Manufacturers",
            "GTM_Opportunity_Score",
            "GTM_Priority",
            "GTM_Strategic_Action"
        ]
    ]
    .sort_values(
        "GTM_Opportunity_Score",
        ascending=False
    )
)

In [ ]:
# ============================================================
# 22. BRAND-LEVEL GTM OPPORTUNITY
# ============================================================

brand_gtm = (
    gtm_df
    .groupby("Brand Name")
    .agg(
        Sales_Value=("Sales Value", "sum"),
        Sales_Units=("Sales Units", "sum"),
        Channels=("Distribution Channel", "nunique"),
        Manufacturers=("Manufacturer", "nunique"),
        Therapeutic_Classes=("Therapeutic Class", "nunique"),
        Avg_Selling_Price=("Selling Price", "mean"),
        First_Launch=("Product Launch", "min")
    )
    .reset_index()
)

# ------------------------------------------------------------
# Channel reach
# ------------------------------------------------------------

brand_gtm["Channel_Reach_Score"] = minmax(
    brand_gtm["Channels"]
)

# ------------------------------------------------------------
# Commercial scale
# ------------------------------------------------------------

brand_gtm["Sales_Scale_Score"] = minmax(
    brand_gtm["Sales_Value"]
)

# ------------------------------------------------------------
# Unit performance
# ------------------------------------------------------------

brand_gtm["Unit_Scale_Score"] = minmax(
    brand_gtm["Sales_Units"]
)

# ------------------------------------------------------------
# GTM breadth
# ------------------------------------------------------------

brand_gtm["GTM_Breadth_Score"] = minmax(
    brand_gtm["Channels"]
)

# ------------------------------------------------------------
# Final GTM score
# ------------------------------------------------------------

brand_gtm["GTM_Opportunity_Score"] = (
    brand_gtm["Sales_Scale_Score"] * 0.40
    + brand_gtm["Unit_Scale_Score"] * 0.20
    + brand_gtm["Channel_Reach_Score"] * 0.25
    + brand_gtm["GTM_Breadth_Score"] * 0.15
)

display(
    brand_gtm.sort_values(
        "GTM_Opportunity_Score",
        ascending=False
    ).head(50)
)

In [ ]:
# ============================================================
# 23. BRAND GTM STRATEGIC CLASSIFICATION
# ============================================================

brand_gtm["GTM_Tier"] = pd.cut(
    brand_gtm["GTM_Opportunity_Score"],
    bins=[-np.inf, 25, 50, 75, np.inf],
    labels=[
        "Low Priority",
        "Monitor",
        "Strategic Opportunity",
        "High Priority"
    ]
)

def brand_gtm_action(tier):

    if tier == "High Priority":
        return "Accelerate GTM Expansion"

    elif tier == "Strategic Opportunity":
        return "Expand Channel Coverage"

    elif tier == "Monitor":
        return "Optimize Existing Channels"

    else:
        return "Selective GTM Investment"


brand_gtm["GTM_Action"] = (
    brand_gtm["GTM_Tier"]
    .astype(str)
    .apply(brand_gtm_action)
)

display(
    brand_gtm[
        [
            "Brand Name",
            "Sales_Value",
            "Channels",
            "GTM_Opportunity_Score",
            "GTM_Tier",
            "GTM_Action"
        ]
    ]
    .sort_values(
        "GTM_Opportunity_Score",
        ascending=False
    )
    .head(100)
)

In [ ]:
# ============================================================
# 24. CHANNEL EXPANSION OPPORTUNITIES
# ============================================================

brand_channel_matrix = (
    gtm_df
    .pivot_table(
        index="Brand Name",
        columns="Distribution Channel",
        values="Sales Value",
        aggfunc="sum",
        fill_value=0
    )
)

brand_total_sales = (
    brand_channel_matrix.sum(axis=1)
)

# ------------------------------------------------------------
# Brands with strong sales but limited channel presence
# ------------------------------------------------------------

brand_channel_opportunity = pd.DataFrame({
    "Brand Name": brand_total_sales.index,
    "Total_Sales": brand_total_sales.values,
    "Channel_Count": (
        (brand_channel_matrix > 0)
        .sum(axis=1)
        .values
    )
})

brand_channel_opportunity["Sales_per_Channel"] = (
    brand_channel_opportunity["Total_Sales"]
    / brand_channel_opportunity["Channel_Count"]
)

brand_channel_opportunity = (
    brand_channel_opportunity
    .sort_values(
        "Sales_per_Channel",
        ascending=False
    )
)

display(
    brand_channel_opportunity.head(50)
)

In [ ]:
# ============================================================
# 25. GTM CHANNEL HEATMAP
# ============================================================

heatmap_data = (
    gtm_df
    .pivot_table(
        index="Therapeutic Class",
        columns="Distribution Channel",
        values="Sales Value",
        aggfunc="sum",
        fill_value=0
    )
)

# Keep top therapeutic classes
top_therapeutics = (
    heatmap_data.sum(axis=1)
    .sort_values(ascending=False)
    .head(20)
    .index
)

heatmap_data = heatmap_data.loc[
    top_therapeutics
]

plt.figure(
    figsize=(14, 9)
)

plt.imshow(
    np.log1p(heatmap_data.values),
    aspect="auto"
)

plt.title(
    "Therapeutic Class x Distribution Channel"
)

plt.xlabel("Distribution Channel")
plt.ylabel("Therapeutic Class")

plt.xticks(
    range(len(heatmap_data.columns)),
    heatmap_data.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(heatmap_data.index)),
    heatmap_data.index
)

plt.colorbar(
    label="Log(Sales Value + 1)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 26. CHANNEL vs PRICE STRATEGY MATRIX
# ============================================================

plt.figure(figsize=(10, 7))

plt.scatter(
    channel_strategy["Avg_Selling_Price"],
    channel_strategy["Sales_Value"],
    s=channel_strategy["Brands"] * 5 + 50,
    alpha=0.7
)

for _, row in channel_strategy.iterrows():

    plt.annotate(
        row["Distribution Channel"],
        (
            row["Avg_Selling_Price"],
            row["Sales_Value"]
        ),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.title(
    "Distribution Channel: Price vs Sales Opportunity"
)

plt.xlabel("Average Selling Price")
plt.ylabel("Sales Value")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 27. TOP GTM OPPORTUNITIES
# ============================================================

top_gtm_opportunities = (
    brand_gtm[
        brand_gtm["GTM_Tier"].isin(
            [
                "High Priority",
                "Strategic Opportunity"
            ]
        )
    ]
    .sort_values(
        "GTM_Opportunity_Score",
        ascending=False
    )
    .head(50)
)

display(top_gtm_opportunities)

In [ ]:
# ============================================================
# 28. EXECUTIVE GTM KPIs
# ============================================================

high_priority_channels = (
    channel_strategy[
        channel_strategy["GTM_Priority"]
        == "High Priority"
    ]
    .shape[0]
)

strategic_channels = (
    channel_strategy[
        channel_strategy["GTM_Priority"]
        == "Strategic Opportunity"
    ]
    .shape[0]
)

high_priority_brands = (
    brand_gtm[
        brand_gtm["GTM_Tier"]
        == "High Priority"
    ]
    .shape[0]
)

strategic_brands = (
    brand_gtm[
        brand_gtm["GTM_Tier"]
        == "Strategic Opportunity"
    ]
    .shape[0]
)

gtm_kpis = pd.DataFrame({
    "KPI": [
        "Total Market Sales",
        "Total Brands",
        "Total Channels",
        "Total Manufacturers",
        "High Priority Channels",
        "Strategic Opportunity Channels",
        "High Priority Brands",
        "Strategic Opportunity Brands"
    ],
    "Value": [
        total_sales,
        total_brands,
        total_channels,
        total_manufacturers,
        high_priority_channels,
        strategic_channels,
        high_priority_brands,
        strategic_brands
    ]
})

display(gtm_kpis)

In [ ]:
# ============================================================
# 29. EXECUTIVE GTM DASHBOARD
# ============================================================

fig = plt.figure(
    figsize=(18, 12)
)

# ------------------------------------------------------------
# KPI section
# ------------------------------------------------------------

ax1 = plt.subplot(2, 2, 1)

top_channels = (
    channel_strategy
    .sort_values(
        "Sales_Value",
        ascending=False
    )
    .head(10)
)

ax1.barh(
    top_channels[
        "Distribution Channel"
    ][::-1],
    top_channels[
        "Sales_Value"
    ][::-1]
)

ax1.set_title(
    "Top Distribution Channels by Sales"
)

ax1.set_xlabel("Sales Value")

# ------------------------------------------------------------
# Channel opportunity score
# ------------------------------------------------------------

ax2 = plt.subplot(2, 2, 2)

score_data = (
    channel_strategy
    .sort_values(
        "GTM_Opportunity_Score",
        ascending=False
    )
)

ax2.barh(
    score_data[
        "Distribution Channel"
    ][::-1],
    score_data[
        "GTM_Opportunity_Score"
    ][::-1]
)

ax2.set_title(
    "GTM Opportunity Score by Channel"
)

ax2.set_xlabel("GTM Opportunity Score")

# ------------------------------------------------------------
# Top brands
# ------------------------------------------------------------

ax3 = plt.subplot(2, 2, 3)

top_brands = (
    brand_gtm
    .sort_values(
        "GTM_Opportunity_Score",
        ascending=False
    )
    .head(15)
)

ax3.barh(
    top_brands[
        "Brand Name"
    ][::-1],
    top_brands[
        "GTM_Opportunity_Score"
    ][::-1]
)

ax3.set_title(
    "Top GTM Brand Opportunities"
)

ax3.set_xlabel(
    "GTM Opportunity Score"
)

# ------------------------------------------------------------
# GTM tier distribution
# ------------------------------------------------------------

ax4 = plt.subplot(2, 2, 4)

tier_distribution = (
    brand_gtm["GTM_Tier"]
    .value_counts()
)

ax4.bar(
    tier_distribution.index.astype(str),
    tier_distribution.values
)

ax4.set_title(
    "Brand GTM Strategic Tier Distribution"
)

ax4.set_ylabel(
    "Number of Brands"
)

ax4.tick_params(
    axis="x",
    rotation=45
)

plt.suptitle(
    "PharmaLens AI — Executive GTM Intelligence Dashboard",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 30. GTM STRATEGIC RECOMMENDATIONS
# ============================================================

recommendations = []

# ------------------------------------------------------------
# Channel recommendations
# ------------------------------------------------------------

for _, row in channel_strategy.iterrows():

    channel = row["Distribution Channel"]
    priority = row["GTM_Priority"]
    score = row["GTM_Opportunity_Score"]

    if priority == "High Priority":

        action = (
            f"Scale investment in {channel}. "
            f"This channel has strong commercial scale "
            f"and a high GTM opportunity score ({score:.1f})."
        )

    elif priority == "Strategic Opportunity":

        action = (
            f"Expand and validate {channel}. "
            f"Use targeted brand/channel expansion "
            f"to capture additional growth."
        )

    elif priority == "Monitor":

        action = (
            f"Optimize {channel}. "
            f"Maintain presence while improving "
            f"channel efficiency and brand productivity."
        )

    else:

        action = (
            f"Use selective coverage in {channel}. "
            f"Avoid excessive resource allocation "
            f"unless strategic evidence improves."
        )

    recommendations.append({
        "Type": "Channel",
        "Entity": channel,
        "Priority": str(priority),
        "GTM_Score": score,
        "Recommendation": action
    })

# ------------------------------------------------------------
# Brand recommendations
# ------------------------------------------------------------

for _, row in (
    brand_gtm
    .sort_values(
        "GTM_Opportunity_Score",
        ascending=False
    )
    .head(100)
    .iterrows()
):

    brand = row["Brand Name"]
    tier = row["GTM_Tier"]
    score = row["GTM_Opportunity_Score"]

    if tier == "High Priority":

        action = (
            "Accelerate GTM expansion and evaluate "
            "additional distribution channels."
        )

    elif tier == "Strategic Opportunity":

        action = (
            "Expand channel coverage selectively and "
            "validate incremental commercial potential."
        )

    elif tier == "Monitor":

        action = (
            "Optimize existing channel performance "
            "before significant expansion."
        )

    else:

        action = (
            "Maintain selective coverage and prioritize "
            "higher-return GTM opportunities."
        )

    recommendations.append({
        "Type": "Brand",
        "Entity": brand,
        "Priority": str(tier),
        "GTM_Score": score,
        "Recommendation": action
    })

gtm_recommendations = pd.DataFrame(
    recommendations
)

display(
    gtm_recommendations.head(100)
)

In [ ]:
# ============================================================
# 31. FINAL GTM STRATEGIC SUMMARY
# ============================================================

top_channel = (
    channel_strategy
    .sort_values(
        "Sales_Value",
        ascending=False
    )
    .iloc[0]
)

top_gtm_channel = (
    channel_strategy
    .sort_values(
        "GTM_Opportunity_Score",
        ascending=False
    )
    .iloc[0]
)

top_gtm_brand = (
    brand_gtm
    .sort_values(
        "GTM_Opportunity_Score",
        ascending=False
    )
    .iloc[0]
)

print("=" * 80)
print("PHARMALENS AI — GTM INTELLIGENCE FINAL SUMMARY")
print("=" * 80)

print(
    f"\nTotal Market Sales: "
    f"{total_sales:,.0f}"
)

print(
    f"Total Brands: "
    f"{total_brands:,}"
)

print(
    f"Total Distribution Channels: "
    f"{total_channels:,}"
)

print(
    f"\nLargest Sales Channel: "
    f"{top_channel['Distribution Channel']}"
)

print(
    f"Largest Channel Sales Share: "
    f"{top_channel['Sales_Share_%']:.2f}%"
)

print(
    f"\nHighest GTM Opportunity Channel: "
    f"{top_gtm_channel['Distribution Channel']}"
)

print(
    f"GTM Opportunity Score: "
    f"{top_gtm_channel['GTM_Opportunity_Score']:.2f}"
)

print(
    f"\nHighest GTM Opportunity Brand: "
    f"{top_gtm_brand['Brand Name']}"
)

print(
    f"GTM Opportunity Score: "
    f"{top_gtm_brand['GTM_Opportunity_Score']:.2f}"
)

print(
    f"\nHigh Priority Channels: "
    f"{high_priority_channels}"
)

print(
    f"Strategic Opportunity Channels: "
    f"{strategic_channels}"
)

print(
    f"High Priority Brands: "
    f"{high_priority_brands:,}"
)

print(
    f"Strategic Opportunity Brands: "
    f"{strategic_brands:,}"
)

print("\n" + "=" * 80)
print("KEY GTM MESSAGE")
print("=" * 80)

print(
    "\nThe GTM framework identifies where pharmaceutical "
    "commercial resources should be concentrated by combining "
    "channel scale, brand reach, manufacturer coverage, "
    "unit performance, and channel expansion potential."
)

print(
    "\nThe objective is not simply to identify the largest "
    "distribution channel, but to identify the channels, "
    "brands, and therapeutic segments where incremental "
    "commercial investment has the greatest strategic potential."
)

In [ ]:
# ============================================================
# 32. EXPORT GTM INTELLIGENCE
# ============================================================

OUTPUT_DIR = "../data/gtm_outputs"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Export tables
# ------------------------------------------------------------

channel_intelligence.to_csv(
    f"{OUTPUT_DIR}/channel_intelligence.csv",
    index=False
)

channel_strategy.to_csv(
    f"{OUTPUT_DIR}/channel_gtm_strategy.csv",
    index=False
)

brand_channel.to_csv(
    f"{OUTPUT_DIR}/brand_channel_intelligence.csv",
    index=False
)

manufacturer_channel.to_csv(
    f"{OUTPUT_DIR}/manufacturer_channel_intelligence.csv",
    index=False
)

therapeutic_channel.to_csv(
    f"{OUTPUT_DIR}/therapeutic_channel_intelligence.csv",
    index=False
)

brand_gtm.to_csv(
    f"{OUTPUT_DIR}/brand_gtm_intelligence.csv",
    index=False
)

gtm_recommendations.to_csv(
    f"{OUTPUT_DIR}/gtm_recommendations.csv",
    index=False
)

if not new_product_channel.empty:

    new_product_channel.to_csv(
        f"{OUTPUT_DIR}/new_product_channel_strategy.csv",
        index=False
    )

print("✅ GTM intelligence files exported successfully.")

print(
    f"\nOutput directory:\n{OUTPUT_DIR}"
)

In [ ]:
# ============================================================
# 33. EXPORT COMPLETE GTM EXCEL REPORT
# ============================================================

excel_output = (
    f"{OUTPUT_DIR}/PharmaLens_AI_GTM_Intelligence.xlsx"
)

with pd.ExcelWriter(
    excel_output,
    engine="openpyxl"
) as writer:

    gtm_overview.to_excel(
        writer,
        sheet_name="GTM Overview",
        index=False
    )

    channel_intelligence.to_excel(
        writer,
        sheet_name="Channel Intelligence",
        index=False
    )

    channel_strategy.to_excel(
        writer,
        sheet_name="Channel Strategy",
        index=False
    )

    brand_channel.to_excel(
        writer,
        sheet_name="Brand Channel",
        index=False
    )

    manufacturer_channel.to_excel(
        writer,
        sheet_name="Manufacturer Channel",
        index=False
    )

    therapeutic_channel.to_excel(
        writer,
        sheet_name="Therapeutic Channel",
        index=False
    )

    brand_gtm.to_excel(
        writer,
        sheet_name="Brand GTM",
        index=False
    )

    gtm_recommendations.to_excel(
        writer,
        sheet_name="Recommendations",
        index=False
    )

    if not new_product_channel.empty:

        new_product_channel.to_excel(
            writer,
            sheet_name="New Product GTM",
            index=False
        )

print(
    "✅ Complete GTM Excel report created:"
)

print(excel_output)

In [ ]:
# ============================================================
# 34. FINAL VALIDATION
# ============================================================

print("=" * 80)
print("PHARMALENS AI — 05_GTM_INTELLIGENCE VALIDATION")
print("=" * 80)

validation_items = {
    "Dataset Loaded": len(gtm_df) > 0,
    "Column Validation": len(missing_columns) == 0,
    "Channel Intelligence": len(channel_intelligence) > 0,
    "Brand x Channel": len(brand_channel) > 0,
    "Manufacturer x Channel": len(manufacturer_channel) > 0,
    "Therapeutic x Channel": len(therapeutic_channel) > 0,
    "Brand GTM Scoring": len(brand_gtm) > 0,
    "Channel GTM Scoring": len(channel_strategy) > 0,
    "GTM Recommendations": len(gtm_recommendations) > 0,
    "Excel Export": os.path.exists(excel_output)
}

for item, status in validation_items.items():

    symbol = "✅" if status else "❌"

    print(
        f"{symbol} {item}"
    )

print("=" * 80)

if all(validation_items.values()):

    print(
        "✅ 05_GTM_Intelligence completed successfully."
    )

else:

    print(
        "⚠️ Some GTM components require review."
    )